# Add APIs to Multi-Turn Dialogues
#### Temporary files to add APIs to the multi-turn dialogues. This should be moved to a script once m3_benchmark scripts are setup.

In [30]:
import json
import os
from tqdm import tqdm
from collections import defaultdict
import random
import copy
import pdb

In [31]:
multiturn_foldername="/proj/m3benchmark/m3data/balanced"
output_foldername="/proj/m3benchmark/m3data/0825_balanced_rest_v2"
rest_api_foldername="/proj/m3benchmark/invocable-api-datasets/rest_train/DGT_enriched/v2_0809/train"

## Add REST APIs to multi-turn dialogues

In [32]:
RETRIVER_SOFT_CLUSTERS=[
    {
        "name": "Education & Students",
        "description": "Topics related to schools, universities, student information, and education statistics.",
        "members": [
            "california_schools",
            "student_club",
            "student_loan",
            "university",
            "cs_semester",
            "computer_student",
            "college_completion"
        ]
    },
    {
        "name": "Movies, TV & Entertainment, Social Media",
        "description": "Topics focused on movies, episodes, characters, and streaming platforms.",
        "members": [
            "movie",
            "movie_platform",
            "movie_3",
            "movies_4",
            "simpson_episodes",
            "disney",
            "law_episode",
            "movielens",
            "social_media",
            "talkingdata",
            "music_tracker",
            "music_platform_2",
            "card_games",
            "video_games",
            "superhero"
        ]
    },
    {
        "name": "Sports & Athletics",
        "description": "Topics covering various sports, athletes, games, and statistics from different leagues.",
        "members": [
            "formula_1",
            "professional_basketball",
            "european_football_1",
            "european_football_2",
            "olympics",
            "ice_hockey_draft",
            "hockey",
            "soccer_2016"
        ]
    },
    {
        "name": "Retail, Sales & Commerce",
        "description": "Topics related to retail operations, customer purchases, inventory, and product ordering.",
        "members": [
            "retails",
            "retail_world",
            "retail_complains",
            "superstore",
            "regional_sales",
            "car_retails",
            "book_publishing_company",
            "sales",
            "sales_in_weather",
            "works_cycles"
        ]
    },
    {
        "name": "Food, Restaurants & Beverage",
        "description": "Topics concerning restaurants, food inspection, menu items, and beverages like beer.",
        "members": [
            "restaurant",
            "food_inspection",
            "food_inspection_2",
            "cookbook",
            "beer_factory",
            "craftbeer",
            "menu"
        ]
    },
    {
        "name": "Technology & Software",
        "description": "Topics focusing on codebases, software platforms, applications, and user interactions in tech.",
        "members": [
            "app_store",
            "codebase_community",
            "codebase_comments",
            "software_company",
            "image_and_language"
        ]
    },
    {
        "name": "Health & Medicine",
        "description": "Topics that pertain to health data, medical conditions, surveys, and synthetic patients.",
        "members": [
            "toxicology",
            "synthea",
            "mental_health_survey",
            "thrombosis_prediction",
            "genes"
        ]
    },
    {
        "name": "Geography & Demographics",
        "description": "Topics centered around population, regional information, countries, and life expectancy.",
        "members": [
            "mondial_geo",
            "world",
            "address",
            "world_development_indicators"
        ]
    },
    {
        "name": "Transportation & Mobility",
        "description": "Topics dealing with modes of transport such as trains, cars, bikes, and flights.",
        "members": [
            "airline",
            "trains",
            "bike_share_1",
            "cars",
            "shipping"
        ]
    },
    {
        "name": "Finance & Economics",
        "description": "Topics that revolve around financial data, transactions, donations, and currencies.",
        "members": [
            "financial",
            "donor",
            "coinmarketcap",
            "debit_card_specializing"
        ]
    },
    {
        "name": "Government, Law & Public Services, Human Capital",
        "description": "Topics concerning legislation, public databases, law enforcement, and civic institutions.",
        "members": [
            "legislator",
            "chicago_crime",
            "shooting",
            "public_review_platform",
            "human_resources"
        ]
    },
    {
        "name": "Literature, Language & Publishing",
        "description": "Topics focusing on books, authors, literary works, and linguistic corpora.",
        "members": [
            "books",
            "authors",
            "shakespeare",
            "language_corpus",
            "citeseer"
        ]
    }
]

In [34]:
def process_soft_clusters():
    domain_retrievers=defaultdict(list)
    for idx, cluster in enumerate(RETRIVER_SOFT_CLUSTERS):
        potential_clusters = copy.deepcopy(RETRIVER_SOFT_CLUSTERS)
        candidate_cluster = potential_clusters[:idx]+potential_clusters[idx+1:]
        for domain in cluster["members"]:
            candidate_clusters = [i["members"] for i in random.sample(candidate_cluster, k=4)] # select 4 candidate clusters
            possible_domains = random.sample([item for sublist in candidate_clusters for item in sublist], k=9) # Sample 9 domains except for the main domain
            possible_domains.append(domain)
            random.shuffle(possible_domains)
            domain_retrievers[domain]=["clapnq-"+str(i) for i in possible_domains]
    return domain_retrievers

def convert_apis_dict(apis):
    ques_id_dict={}
    id_elem_dict={}
    for api in apis:
        ques_id_dict[api["input"]]=api["sample_id"]
        id_elem_dict[api["sample_id"]]=api
    return ques_id_dict, id_elem_dict

def add_retriever_output(db_id, query):
    RETRIEVAL_OUTPUT={
        "name": "retriever_clapnq_"+str(db_id),
        "arguments": {
            # "collection": "clapnq_"+str(db_id),
            "query": query
        }
    }
    return [RETRIEVAL_OUTPUT]

def check_tool_list(tools_created, retrievers, apis):
    tool_names=[]
    tool_names_created=[tools_created[item]["name"] for item in range(len(tools_created))]
    for api in apis:
        tool_names.append(api["name"])
    for ret in retrievers:
        tool_names.append(ret.replace("clapnq-","retriever_clapnq_"))
    if len(set(tool_names).difference(set(tool_names_created))) == 0:
        return True
    else:
        return False


In [ ]:
filenames=os.listdir(multiturn_foldername)
for filename in tqdm(filenames):
    domain=filename.split("_multiturn_bird.json")[0]
    api_filename=f"en_mixtral-8x22b_{domain}_nestful_format_bird.json"
    with open(f"{multiturn_foldername}/{filename}", 'r') as j:
        data = json.load(j)
    with open(f"{rest_api_foldername}/{api_filename}", 'r') as j:
        apis = json.load(j)
    ques_id_dict, id_elem_dict=convert_apis_dict(apis)
    for idx_item, item in enumerate(data):
        retrievers_per_domain = process_soft_clusters()
        tool_list=[]
        for idx_turn, turn in enumerate(item["turns"]):
            for idx_hop, hop in enumerate(turn["gold_sequence"]):
                if hop["question_type"] == "API":
                    if hop["question"] not in ques_id_dict:
                        if "unanswerable" in data[idx_item].keys():
                            data[idx_item]["unanswerable"] = data[idx_item]["unanswerable"]+f" || No API for turn {idx_turn+1} hop {idx_hop+1}"
                        else:
                            data[idx_item].update({"unanswerable":f"No API for turn {idx_turn+1} hop {idx_hop+1}"})
                        continue
                    sample_id=ques_id_dict[hop["question"]]
                    api_elem=copy.deepcopy(id_elem_dict[sample_id])
                    assert api_elem["input"]==hop["question"]
                    if api_elem["ignore"]:
                        if "unanswerable" in data[idx_item].keys():
                            data[idx_item]["unanswerable"] = data[idx_item]["unanswerable"]+f" || Ignore flag is set to true for API for turn {idx_turn+1} hop {idx_hop+1}"
                        else:
                            data[idx_item].update({"unanswerable":f"Ignore flag is set to true for API for turn {idx_turn+1} hop {idx_hop+1}"})
                        continue
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["OUTPUT_AFTER_EXECUTING_API"]=api_elem["OUTPUT_AFTER_EXECUTING_API"]
                    # data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["tools"]=api_elem["tools"] # Common tool list added
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["API"]=api_elem["API"]                    
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["output"]=api_elem["output"]
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["ignore"]=api_elem["ignore"]
                    data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["parameterized_query"]=api_elem["parameterized_query"]
                    if len(tool_list)!=0:
                        if not set([tl["name"] for tl in tool_list]) == set([el["name"] for el in api_elem["tools"]]):
                            # These should be same.
                            import pdb
                            pdb.set_trace()
                    elif len(tool_list)==0:
                        tool_list=api_elem["tools"]

        data[idx_item]["retrievers"]=retrievers_per_domain[item["dataset_name"]]
        if "clapnq-"+item["dataset_name"] not in data[idx_item]["retrievers"]:
            pdb.set_trace()
            data[idx_item]["retrievers"].append("clapnq-"+item["dataset_name"]) # Adding base dataset name if not present

        # Add retrivers to tool list
        ret_names=["retriever_clapnq_"+i.split("clapnq-")[1] for i in data[idx_item]["retrievers"]]
        retrieval_tools=[]
        for ret_name in ret_names:
            retrieval_tool = {
            "name": ret_name,
            "description": "Retrieve document(s) that best matches the query related to "+" ".join(ret_name.split("clapnq_")[-1].split("_")),
            "parameters": {
                "type": "object",
                # "required": ["collection", "query"],                
                "required": ["query"],
                "properties": {
                    # "collection": {
                    #     "description": "Name of the collection to retrieve documents from.",
                    #     "type": "string",
                    #     "enum": ret_names,
                    # },
                    "query": {
                        "description": "Query for retrieving the document(s).",
                        "type": "string"
                    }
                }
            }
            }
            retrieval_tools.append(retrieval_tool)

        # Add retrievers to the output
        if "RAG" in item["type"]:
            for turn_idx, turn in enumerate(item["turns"]):
                if "RAG" in turn["type"]:
                    for seq_idx, sequence in enumerate(turn["gold_sequence"]):
                        if "RAG" in sequence["question_type"]:
                            data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["output"]=add_retriever_output(data[idx_item]["dataset_name"],data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["question"])
                            data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["OUTPUT_AFTER_EXECUTING_API"]=data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["rag_doc"]  # Fix this using "id" and "text"          
        if len(tool_list)!=0:
            data[idx_item]["tools"]=tool_list
            data[idx_item]["tools"].extend(retrieval_tools)
            assert check_tool_list(data[idx_item]["tools"],ret_names,api_elem["tools"])            
        else:
            if "unanswerable" in data[idx_item].keys():
                data[idx_item]["tools"]=[]
            else:
                pdb.set_trace()        

    with open(f'{output_foldername}/{filename}', 'w') as fout:
        json.dump(data , fout)

  0%|          | 0/47 [00:00<?, ?it/s]

100%|██████████| 47/47 [16:42<00:00, 21.33s/it] 


In [ ]:
# filenames=os.listdir(multiturn_foldername)
# # retrievers_per_domain = process_soft_clusters()
# for filename in tqdm(filenames):
#     domain=filename.split("_multiturn_bird.json")[0]
#     api_filename=f"en_mixtral-8x22b_{domain}_nestful_format_bird.json"
#     with open(f"{multiturn_foldername}/{filename}", 'r') as j:
#         data = json.load(j)
#     with open(f"{rest_api_foldername}/{api_filename}", 'r') as j:
#         apis = json.load(j)
#     ques_id_dict, id_elem_dict=convert_apis_dict(apis)
#     for idx_item, item in enumerate(data):
#         retrievers_per_domain = process_soft_clusters() # A different retriever for every data point
#         tool_list=[]
#         for idx_turn, turn in enumerate(item["turns"]):
#             # if isinstance(turn["gold_sequence"][0],list):
#             #     if len(turn["gold_sequence"]) == 1:
#             #         turn["gold_sequence"]=turn["gold_sequence"][0]
#             #     else:
#             #         import pdb
#             #         pdb.set_trace()
#             for idx_hop, hop in enumerate(turn["gold_sequence"]):
#                 if hop["question_type"] == "API":
#                     if hop["sample_id"] not in id_elem_dict.keys():
#                         if "unanswerable" in data[idx_item].keys():
#                             data[idx_item]["unanswerable"] = data[idx_item]["unanswerable"]+f" || No API for turn {idx_turn+1} hop {idx_hop+1}"
#                         else:
#                             data[idx_item].update({"unanswerable":f"No API for turn {idx_turn+1} hop {idx_hop+1}"})
#                         continue                    
                    
#                     if hop["question"] not in ques_id_dict:
#                         data[idx_item].update({"unanswerable":f"No API for turn {idx_turn+1} hop {idx_hop+1}"})
#                         continue
#                     sample_id=ques_id_dict[hop["question"]]
#                     api_elem=copy.deepcopy(id_elem_dict[sample_id])
#                     assert api_elem["input"]==hop["question"]
#                     data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["OUTPUT_AFTER_EXECUTING_API"]=api_elem["OUTPUT_AFTER_EXECUTING_API"]
#                     # data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["tools"]=api_elem["tools"] # Common tool list added
#                     data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["API"]=api_elem["API"]
#                     data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["output"]=api_elem["output"]
#                     data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["ignore"]=api_elem["ignore"]
#                     if api_elem["ignore"]=="true":
#                         data[idx_item].update({"unanswerable":f"Ignore flag set to true on API for turn {idx_turn+1} hop {idx_hop+1}"})
#                         continue
#                     data[idx_item]["turns"][idx_turn]["gold_sequence"][idx_hop]["parameterized_query"]=api_elem["parameterized_query"]
#                     if len(tool_list)!=0:
#                         if not set([tl["name"] for tl in tool_list]) == set([el["name"] for el in api_elem["tools"]]):
#                             # These should be same.
#                             import pdb
#                             pdb.set_trace()
#                     elif len(tool_list)==0:
#                         tool_list=api_elem["tools"]

#         data[idx_item]["retrievers"]=retrievers_per_domain[item["dataset_name"]]

#         # Add retrivers to tool list
#         ret_names=["retriever_clapnq_"+i.split("clapnq-")[1] for i in retrievers_per_domain[item["dataset_name"]]]
#         retrieval_tools=[]
#         for ret_name in ret_names:
#             retrieval_tool = {
#             "name": ret_name,
#             "description": "Retrieve document(s) that best matches the query related to "+" ".join(ret_name.split("clapnq_")[-1].split("_")),
#             "parameters": {
#                 "type": "object",
#                 # "required": ["collection", "query"],                
#                 "required": ["query"],
#                 "properties": {
#                     # "collection": {
#                     #     "description": "Name of the collection to retrieve documents from.",
#                     #     "type": "string",
#                     #     "enum": ret_names,
#                     # },
#                     "query": {
#                         "description": "Query for retrieving the document(s).",
#                         "type": "string"
#                     }
#                 }
#             }
#             }
#             retrieval_tools.append(retrieval_tool)

#         # Add retrievers to the output
#         if "RAG" in item["type"]:
#             for turn_idx, turn in enumerate(item["turns"]):
#                 if "RAG" in turn["type"]:
#                     for seq_idx, sequence in enumerate(turn["gold_sequence"]):
#                         if "RAG" in sequence["question_type"]:
#                             data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["output"]=add_retriever_output(data[idx_item]["dataset_name"],data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["question"])
#                             data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["OUTPUT_AFTER_EXECUTING_API"]=data[idx_item]["turns"][turn_idx]["gold_sequence"][seq_idx]["rag_doc"]  # Fix this using "id" and "text"          
#         if len(tool_list)!=0:
#             data[idx_item]["tools"]=tool_list
#             data[idx_item]["tools"].append(retrieval_tools)
#         else:
#             if "unanswerable" in data[idx_item].keys():
#                 data[idx_item]["tools"]=[]
#             else:
#                 pdb.set_trace()

#         # assert check_tool_list(data[idx_item]["tools"],ret_names,api_elem["tools"])

#     with open(f'{output_foldername}/{filename}', 'w') as fout:
#         json.dump(data , fout)

 79%|███████▊  | 37/47 [05:10<01:23,  8.39s/it]


FileNotFoundError: [Errno 2] No such file or directory: '/proj/m3benchmark/m3data/0825_balanced_rest_v2/ice_hockey_draft_multiturn_bird.json'

In [38]:
count_unanaswerable=defaultdict(int)
total_count=defaultdict(int)
filenames=os.listdir(output_foldername)
for filename in tqdm(filenames):
    domain=filename.split("_multiturn_bird.json")[0]
    with open(f"{output_foldername}/{filename}", 'r') as j:
        data = json.load(j)
    for idx_item, item in enumerate(data):
        total_count["domain"]+=1
        if "unanswerable" in item.keys():
            count_unanaswerable[domain]+=1

100%|██████████| 47/47 [02:43<00:00,  3.47s/it]


In [39]:
sum(count_unanaswerable.values()), sum(total_count.values())

(15722, 31953)

0.49203517666572777